# 🥔 EU Potato Supply Chain — Economics & Data Analytics

**Author:** Tejaswini Sengaonkar  
**Sources:** Eurostat (apro_cpsh1, apri_pi15_outa) · World Bank Pink Sheet · FADN / Copa-Cogeca  
**Markets:** France · Netherlands · Poland · Germany · 2015–2023  

---

This notebook analyses on-farm production economics across Europe's four largest potato markets,
evaluates data source quality, and models the economic impact of regenerative agriculture practices.
It directly maps to objectives relevant to global food companies managing European agricultural supply chains.

**Run all cells in order** (`Kernel → Restart & Run All`).


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.1f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# Paths — adjust if running notebook from a different directory
DATA_RAW = os.path.join('data', 'raw')
DATA_PROC = os.path.join('data', 'processed')
OUTPUTS   = 'outputs'
os.makedirs(DATA_PROC, exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)

COUNTRY_COLORS = {
    'France': '#003189',
    'Netherlands': '#E77900',
    'Poland': '#C0392B',
    'Germany': '#555555',
}

print("✓ Setup complete")
print(f"  pandas {pd.__version__} | matplotlib {matplotlib.__version__}")


## 1. Load Real Data

Data loaded from government open-data sources:
- **Eurostat `apro_cpsh1`** — area harvested (ha) and production (tonnes), CC BY 4.0
- **Eurostat `apri_pi15_outa`** — potato producer price indices (2015=100), CC BY 4.0  
- **World Bank Pink Sheet** — urea and energy price indices, CC BY 4.0
- **FADN / Copa-Cogeca / WEcR** — published input cost benchmarks per country


In [ ]:
# Load all data sources
production = pd.read_csv(os.path.join(DATA_RAW, 'eurostat_production_verified.csv'), comment='#')
price_idx  = pd.read_csv(os.path.join(DATA_RAW, 'eurostat_price_index_verified.csv'), comment='#')
commodities= pd.read_csv(os.path.join(DATA_RAW, 'worldbank_commodity_prices_verified.csv'), comment='#')
costs      = pd.read_csv(os.path.join(DATA_RAW, 'fadn_input_costs_verified.csv'), comment='#')
sources    = pd.read_csv(os.path.join(DATA_RAW, 'data_source_inventory.csv'))
regen      = pd.read_csv(os.path.join(DATA_RAW, 'regen_practices.csv'))

print(f"Production data:  {production.shape} | Years {production.year.min()}–{production.year.max()}")
print(f"Price index:      {price_idx.shape}")
print(f"Commodity prices: {commodities.shape}")
print(f"Input costs:      {costs.shape}")
print(f"Data sources:     {sources.shape[0]} evaluated")
print(f"Regen practices:  {regen.shape[0]} practices")
production.head(8)


## 2. Merge & Calculate Economics

In [ ]:
# Merge all datasets
df = pd.merge(production, costs[['country','year','total_variable_eur_ha',
    'fertilizer_eur_ha','labour_eur_ha','land_rent_eur_ha','seed_eur_ha',
    'pesticide_eur_ha','fuel_energy_eur_ha','machinery_eur_ha','irrigation_eur_ha']],
    on=['country','year'], how='left')

df = pd.merge(df, price_idx, on=['country','year'], how='left')
df = pd.merge(df, commodities[['year','urea_usd_mt','natural_gas_idx_2015_100']], on='year', how='left')

# Estimated farmgate price from price index
base_prices = {'France': 130, 'Netherlands': 145, 'Poland': 95, 'Germany': 125}
df['base_price_eur_t']    = df['country'].map(base_prices)
df['farmgate_price_eur_t'] = (df['base_price_eur_t'] * df['price_index_2015_100'] / 100).round(1)

# Economics
df['revenue_eur_ha']      = (df['yield_t_ha'] * df['farmgate_price_eur_t']).round(0)
df['gross_margin_eur_ha'] = (df['revenue_eur_ha'] - df['total_variable_eur_ha']).round(0)
df['cost_per_tonne_eur']  = (df['total_variable_eur_ha'] / df['yield_t_ha']).round(1)
df['margin_pct']          = (df['gross_margin_eur_ha'] / df['revenue_eur_ha'] * 100).round(1)

# Cost shares
df['fertilizer_share_pct'] = (df['fertilizer_eur_ha'] / df['total_variable_eur_ha'] * 100).round(1)
df['labour_share_pct']     = (df['labour_eur_ha']      / df['total_variable_eur_ha'] * 100).round(1)
df['land_share_pct']       = (df['land_rent_eur_ha']   / df['total_variable_eur_ha'] * 100).round(1)

# Fertilizer cost index (2019 = 100 per country)
base_fert = df[df['year']==2019].set_index('country')['fertilizer_eur_ha']
df['fert_idx_2019'] = df.apply(
    lambda r: round(r['fertilizer_eur_ha'] / base_fert.get(r['country'], np.nan) * 100, 1), axis=1)

df.to_csv(os.path.join(DATA_PROC, 'economics_merged.csv'), index=False)
print(f"✓ Merged dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df[['country','year','yield_t_ha','farmgate_price_eur_t','total_variable_eur_ha',
    'gross_margin_eur_ha','cost_per_tonne_eur','margin_pct']].tail(8)


## 3. Input Cost Structure by Country (2023)

In [ ]:
latest = df[df['year'] == 2023].set_index('country')
cost_cols = {
    'Seed': 'seed_eur_ha', 'Fertilizer': 'fertilizer_eur_ha',
    'Pesticides': 'pesticide_eur_ha', 'Fuel/Energy': 'fuel_energy_eur_ha',
    'Labour': 'labour_eur_ha', 'Machinery': 'machinery_eur_ha',
    'Land Rent': 'land_rent_eur_ha', 'Irrigation': 'irrigation_eur_ha',
}
countries = ['Poland', 'Germany', 'France', 'Netherlands']
data = latest.loc[countries, list(cost_cols.values())]
data.columns = list(cost_cols.keys())

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#264653','#2A9D8F','#E9C46A','#F4A261','#E76F51','#8338EC','#3A86FF','#FB5607']
bottom = np.zeros(len(countries))

for (label, color) in zip(cost_cols.keys(), colors):
    ax.barh(countries, data[label].values, left=bottom, color=color, label=label, height=0.55)
    bottom += data[label].values

for i, country in enumerate(countries):
    total = data.loc[country].sum()
    ax.text(total + 40, i, f'€{total:,.0f}/ha', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('EUR / hectare', fontsize=10)
ax.set_title('Potato Input Cost Breakdown by Country (2023)\nSource: FADN / Copa-Cogeca / WEcR', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=8, ncol=2, framealpha=0.8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS, 'cost_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n2023 Cost Summary:")
print(data.assign(Total=data.sum(axis=1)).to_string())


## 4. Production Cost per Tonne (2015–2023)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cost per tonne
ax = axes[0]
for country in ['France','Netherlands','Poland','Germany']:
    d = df[df['country']==country].sort_values('year')
    ax.plot(d['year'], d['cost_per_tonne_eur'], marker='o', linewidth=2,
            color=COUNTRY_COLORS[country], label=country)
    ax.annotate(f"€{d['cost_per_tonne_eur'].iloc[-1]:.0f}",
                xy=(d['year'].iloc[-1], d['cost_per_tonne_eur'].iloc[-1]),
                xytext=(5, 0), textcoords='offset points',
                fontsize=8, color=COUNTRY_COLORS[country], fontweight='bold')

ax.set_title('Cost of Production (EUR/tonne)\nSource: Eurostat apro_cpsh1 + FADN costs', fontweight='bold')
ax.set_ylabel('EUR / tonne'); ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:.0f}'))
ax.set_xticks(df['year'].unique()); ax.spines[['top','right']].set_visible(False)

# Right: fertilizer cost index vs urea price
ax2 = axes[1]
for country in ['France','Netherlands','Poland','Germany']:
    d = df[df['country']==country].sort_values('year')
    ax2.plot(d['year'], d['fert_idx_2019'], marker='s', linewidth=2,
             color=COUNTRY_COLORS[country], label=f'{country} (fertilizer cost idx)')

urea = df[['year','urea_usd_mt']].drop_duplicates().sort_values('year')
urea_idx = urea['urea_usd_mt'] / urea[urea['year']==2019]['urea_usd_mt'].values[0] * 100
ax2.plot(urea['year'], urea_idx, 'k--', linewidth=1.5, label='Global Urea Price (WB, 2019=100)')
ax2.axhline(100, color='#aaa', linestyle=':', linewidth=1)
ax2.fill_between([2021,2022], [60,60],[280,280], alpha=0.07, color='red')
ax2.text(2021.5, 268, 'Energy crisis', fontsize=7.5, color='#c0392b', ha='center')
ax2.set_title('Fertilizer Cost Index (2019=100)\nSource: FADN costs + World Bank Pink Sheet', fontweight='bold')
ax2.set_ylabel('Index (2019=100)'); ax2.legend(fontsize=7.5)
ax2.set_xticks(df['year'].unique()); ax2.spines[['top','right']].set_visible(False)

plt.suptitle('EU Potato Production Economics 2015–2023', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS, 'cost_trends.png'), dpi=150, bbox_inches='tight')
plt.show()


## 5. Profitability Ranking (3-Year Average 2021–2023)

In [ ]:
recent = df[df['year'] >= 2021]
ranking = recent.groupby('country').agg(
    avg_cost_per_tonne  =('cost_per_tonne_eur','mean'),
    avg_gross_margin    =('gross_margin_eur_ha','mean'),
    avg_margin_pct      =('margin_pct','mean'),
    avg_yield_t_ha      =('yield_t_ha','mean'),
    avg_total_cost      =('total_variable_eur_ha','mean'),
    avg_area_ha         =('area_ha','mean'),
).round(1).sort_values('avg_cost_per_tonne')

ranking['cost_rank']   = ranking['avg_cost_per_tonne'].rank().astype(int)
ranking['margin_rank'] = ranking['avg_gross_margin'].rank(ascending=False).astype(int)

ranking.to_csv(os.path.join(OUTPUTS,'profitability_ranking.csv'))
print("Profitability Ranking (avg 2021-2023):")
print(ranking.to_string())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, label, title in [
    (axes[0], 'avg_cost_per_tonne', 'EUR/tonne', 'Average Cost per Tonne (lower = better)'),
    (axes[1], 'avg_gross_margin',   'EUR/ha',    'Average Gross Margin/ha (higher = better)'),
]:
    vals = ranking[col]
    colors = [COUNTRY_COLORS[c] for c in vals.index]
    bars = ax.bar(vals.index, vals.values, color=colors, width=0.55, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(label)
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'€{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Market Profitability Comparison (2021–2023 avg) | Source: Eurostat + FADN',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS,'profitability.png'), dpi=150, bbox_inches='tight')
plt.show()


## 6. Data Source Inventory & Prioritisation

In [ ]:
quality_map   = {'High': 3, 'Medium': 2, 'Low': 1}
access_map    = {'Open': 3, 'Restricted': 1}
relevance_map = {'High': 3, 'Medium': 2, 'Low': 1}

sources['quality_score']   = sources['data_quality'].map(quality_map)
sources['access_score']    = sources['access_type'].map(access_map)
sources['relevance_score'] = sources['relevance_to_mccain'].map(relevance_map)
sources['composite_score'] = (
    sources['quality_score']   * 0.4 +
    sources['access_score']    * 0.3 +
    sources['relevance_score'] * 0.3
).round(2)
sources['priority_tier'] = pd.cut(
    sources['composite_score'],
    bins=[0, 1.5, 2.2, 3.0],
    labels=['Low Priority', 'Monitor', 'Primary Source']
)
sources.sort_values('composite_score', ascending=False, inplace=True)
sources.to_csv(os.path.join(DATA_PROC,'data_sources_scored.csv'), index=False)

# Priority matrix scatter
fig, ax = plt.subplots(figsize=(9, 6))
tier_colors = {'Primary Source': '#2D6A4F', 'Monitor': '#E9C46A', 'Low Priority': '#E76F51'}
for tier in ['Primary Source','Monitor','Low Priority']:
    sub = sources[sources['priority_tier']==tier]
    ax.scatter(sub['access_score'], sub['quality_score'],
               s=sub['relevance_score']*140, color=tier_colors[tier],
               alpha=0.8, edgecolors='white', linewidth=1.5, label=tier, zorder=5)

for _, row in sources.iterrows():
    name = row['source_name'][:22] + ('…' if len(row['source_name'])>22 else '')
    ax.annotate(name, xy=(row['access_score'], row['quality_score']),
                xytext=(5, 3), textcoords='offset points', fontsize=6.5, alpha=0.8)

ax.set_xlabel('Access Score  (Open=3, Restricted=1)', fontsize=10)
ax.set_ylabel('Data Quality Score (High=3)', fontsize=10)
ax.set_title('Data Source Priority Matrix\n(bubble size = relevance to supply chain decisions)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.9)
ax.set_xlim(0.5, 3.7); ax.set_ylim(0.5, 3.7)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS,'data_source_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nTop Primary Sources:")
print(sources[sources['priority_tier']=='Primary Source'][
    ['source_name','data_type','access_type','composite_score','gaps_identified']
].to_string(index=False))


## 7. Regenerative Agriculture — Economic Opportunity

In [ ]:
col_map = {
    'France':      'applicability_france',
    'Netherlands': 'applicability_netherlands',
    'Poland':      'applicability_poland',
    'Germany':     'applicability_germany',
}
adoption = {'High': 0.30, 'Medium': 0.15, 'Low': 0.05}

rows = []
for country, col in col_map.items():
    latest_row = df[df['country']==country].sort_values('year').iloc[-1]
    area = latest_row['area_ha']
    for _, p in regen.iterrows():
        rate = adoption.get(p[col], 0)
        adopted_ha = area * rate
        saving = p['cost_change_eur_ha'] * adopted_ha * -1
        rows.append({
            'country': country, 'practice': p['practice'], 'category': p['category'],
            'applicability': p[col], 'adopted_ha': round(adopted_ha),
            'cost_impact_eur_ha': p['cost_change_eur_ha'],
            'market_saving_eur': round(saving),
            'carbon_tco2_yr': round(p['carbon_sequestration_tco2_ha_yr'] * adopted_ha, 1),
            'payback_years': p['payback_years'],
        })

regen_df = pd.DataFrame(rows)
regen_df.to_csv(os.path.join(DATA_PROC,'regen_market_impact.csv'), index=False)

summary = regen_df.groupby(['practice','category']).agg(
    total_saving_eur=('market_saving_eur','sum'),
    total_carbon_tco2=('carbon_tco2_yr','sum'),
    avg_payback=('payback_years','mean'),
).reset_index().sort_values('total_saving_eur', ascending=False)

# Chart
fig, ax = plt.subplots(figsize=(12, 5))
colors_regen = ['#2D6A4F' if v > 0 else '#C0392B' for v in summary['total_saving_eur']]
bars = ax.barh(summary['practice'], summary['total_saving_eur']/1e6, color=colors_regen, height=0.55)
ax.axvline(0, color='#999', linewidth=0.8)
ax.set_xlabel('Total Market Saving Potential (EUR million)', fontsize=10)
ax.set_title('Regenerative Agriculture — Estimated Cost Savings Across 4 Markets\n'
             '(30% adoption for high-applicability markets | Source: Goffart et al. 2022, Copa-Cogeca)',
             fontsize=11, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:.1f}M'))
for bar, val in zip(bars, summary['total_saving_eur']):
    xpos = bar.get_width() + (0.15 if val >= 0 else -0.15)
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'€{val/1e6:.1f}M', va='center', ha='left' if val>=0 else 'right', fontsize=8)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS,'regen_savings.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 Regen Opportunities:")
print(summary.head(5)[['practice','total_saving_eur','total_carbon_tco2','avg_payback']].to_string(index=False))


## 8. Key Insights & Recommendations

In [ ]:
latest = df[df['year']==2023]

insights = {
    "Cost competitiveness": (
        f"Poland (€{latest[latest['country']=='Poland']['cost_per_tonne_eur'].values[0]:.0f}/t) "
        f"and Germany (€{latest[latest['country']=='Germany']['cost_per_tonne_eur'].values[0]:.0f}/t) "
        f"are the most cost-efficient markets. Netherlands (€{latest[latest['country']=='Netherlands']['cost_per_tonne_eur'].values[0]:.0f}/t) "
        f"is the least cost-efficient but delivers the highest yield and margin per hectare."
    ),
    "Fertilizer risk": (
        "The 2021–2022 European gas crisis drove urea prices up ~170% (World Bank), "
        "directly increasing fertilizer costs 45–55% above 2019 levels across all markets. "
        "This remains the single largest input cost volatility risk for grower profitability."
    ),
    "Top regen opportunity": (
        f"Precision fertilization offers ~€16.4M in combined market savings at 30% adoption "
        f"across all 4 markets, with a 4-year payback. Combined with cover cropping (3-yr payback), "
        f"these are the most financially accessible entry points to regenerative transition."
    ),
    "Data gap": (
        "No single open-access source provides harmonised, annual, country-level input cost "
        "data for potatoes across FR/NL/PL/DE. Bridging Eurostat production data with FADN "
        "microdata access (requires NDA) is the key analytical gap for ongoing monitoring."
    ),
    "Supply chain recommendation": (
        "For cost-resilient sourcing, Poland offers the lowest cost base but with lower yields. "
        "Germany offers the best balance of volume + margin. The Netherlands commands premium "
        "yields but requires higher farmgate prices to remain viable for growers."
    ),
}

for title, text in insights.items():
    print(f"▶ {title}")
    print(f"  {text}")
    print()

# Save to file
with open(os.path.join(OUTPUTS, 'key_insights.txt'), 'w') as f:
    f.write("EU Potato Supply Chain — Key Insights\n" + "="*50 + "\n\n")
    for title, text in insights.items():
        f.write(f"{title}:\n{text}\n\n")
print("✓ Insights saved to outputs/key_insights.txt")


## 9. Full Dashboard (Summary View)

In [ ]:
# Re-run dashboard script to generate the combined PNG
import subprocess, sys
result = subprocess.run([sys.executable, os.path.join('scripts', '03_dashboard.py')],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode == 0:
    from IPython.display import Image
    display(Image(filename=os.path.join(OUTPUTS, 'dashboard.png'), width=1100))
else:
    print("Dashboard error:", result.stderr)


## 10. (Optional) Live API Fetch from Eurostat & World Bank

Run the cell below to fetch the latest data directly from government APIs.
This will overwrite the `*_verified.csv` fallback files with live data.
**Requires internet access.**


In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, os.path.join('scripts', '00_fetch_data.py')],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Errors:", result.stderr[:500])
